# Clusters image generation smoke tests

This notebook tests `TINTOlib.clusters.Clusters` against the two datasets shipped in `Dataset`: Iris for classification and Boston for regression. Each dataset has its own JSON config under `examples/configs/clusters`, and each config covers every `Clusters.__init__` parameter at least once.

In [ ]:
from pathlib import Path
import inspect
import json
import random
import shutil
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, start / "TINTOlib"]
    for parent in start.parents:
        candidates.extend([parent, parent / "TINTOlib"])

    for candidate in candidates:
        if (candidate / "Dataset").exists() and (candidate / "TINTOlib").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find the TINTOlib repository root from the current working directory.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from TINTOlib.clusters import Clusters

SEED = 64
random.seed(SEED)
np.random.seed(SEED)

CONFIG_DIR = REPO_ROOT / "examples" / "configs" / "clusters"
CONFIG_PATHS = [CONFIG_DIR / "iris.json", CONFIG_DIR / "boston.json"]

RUN_DATASETS = None  # Example: {"iris"}
RUN_CASES = None  # Example: {"kmeans_lloyd_rbf", "mix_method_rgb"}
DRY_RUN = False  # True validates and instantiates models without writing images.

REPO_ROOT, CONFIG_DIR

## Load and validate configs

In [ ]:
CLUSTERS_INIT_PARAMETERS = set(inspect.signature(Clusters.__init__).parameters) - {"self"}


def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def merged_params(config, case):
    params = dict(config.get("base_params", {}))
    params.update(case.get("params", {}))
    return params


def parameter_coverage_rows(config):
    dataset = config["dataset"]
    rows = []
    for parameter in sorted(CLUSTERS_INIT_PARAMETERS):
        case_names = []
        values = []
        for case in config["cases"]:
            params = merged_params(config, case)
            if parameter in params:
                case_names.append(case["name"])
                values.append(params[parameter])

        rows.append({
            "dataset": dataset["name"],
            "parameter": parameter,
            "case_count": len(case_names),
            "cases": ", ".join(case_names),
            "values": json.dumps(values, default=str),
        })
    return rows


def validate_config(config):
    dataset = config["dataset"]
    dataset_path = REPO_ROOT / dataset["path"]
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset file does not exist: {dataset_path}")

    if config.get("base_params", {}).get("problem") != dataset["problem"]:
        raise ValueError(f"base_params.problem must match dataset.problem for {dataset['name']}")

    used_params = set()
    case_names = set()
    for case in config["cases"]:
        if case["name"] in case_names:
            raise ValueError(f"Duplicate case name in {dataset['name']}: {case['name']}")
        case_names.add(case["name"])

        params = merged_params(config, case)
        unknown = set(params) - CLUSTERS_INIT_PARAMETERS
        if unknown:
            raise ValueError(f"Unknown Clusters parameter(s) in {dataset['name']}/{case['name']}: {sorted(unknown)}")
        used_params.update(params)

        # Instantiate here so invalid algorithm-specific combinations fail before generation starts.
        Clusters(**params)

    missing = CLUSTERS_INIT_PARAMETERS - used_params
    if missing:
        raise AssertionError(f"Config for {dataset['name']} does not cover parameter(s): {sorted(missing)}")

    return {
        "dataset": dataset["name"],
        "cases": len(config["cases"]),
        "covered_parameters": len(used_params),
        "required_parameters": len(CLUSTERS_INIT_PARAMETERS),
    }


configs = [load_config(path) for path in CONFIG_PATHS]
coverage = [validate_config(config) for config in configs]
parameter_coverage = pd.DataFrame(
    row
    for config in configs
    for row in parameter_coverage_rows(config)
)

if (parameter_coverage["case_count"] == 0).any():
    missing_rows = parameter_coverage[parameter_coverage["case_count"] == 0]
    raise AssertionError("Missing Clusters parameter coverage:\n" + missing_rows.to_string())

probe = Clusters(problem="classification", normalize=True, verbose=False)
if "normalize" in CLUSTERS_INIT_PARAMETERS and not hasattr(probe, "normalize"):
    print("Note: Clusters accepts normalize, but the current implementation does not store it on the instance.")

pd.DataFrame(coverage)

## Parameter coverage detail


In [ ]:
parameter_coverage.sort_values(["parameter", "dataset"])


## Dataset helpers

In [ ]:
def load_dataset(config):
    dataset = config["dataset"]
    path = REPO_ROOT / dataset["path"]
    target = dataset["target"]

    df = pd.read_csv(path)
    if target not in df.columns:
        raise ValueError(f"Target column {target!r} was not found in {path}")

    feature_cols = [col for col in df.columns if col != target]
    df = df[feature_cols + [target]].copy()

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="raise")

    if not pd.api.types.is_numeric_dtype(df[target]):
        codes, uniques = pd.factorize(df[target])
        df[target] = codes.astype(int)
        print(f"Encoded target mapping for {dataset['name']}: {dict(enumerate(uniques))}")
    else:
        df[target] = pd.to_numeric(df[target], errors="raise")

    if df.isna().any().any():
        missing = df.columns[df.isna().any()].tolist()
        raise ValueError(f"Missing values are not accepted by Clusters. Columns with missing values: {missing}")

    sample = dataset.get("sample", {})
    strategy = sample.get("strategy")
    if strategy == "stratified":
        rows_per_class = int(sample["rows_per_class"])
        sampled_parts = []
        for _, group in df.groupby(target):
            sampled_parts.append(group.sample(n=min(rows_per_class, len(group)), random_state=SEED))
        df = pd.concat(sampled_parts, axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    elif strategy == "random":
        n_rows = min(int(sample["n_rows"]), len(df))
        df = df.sample(n=n_rows, random_state=SEED).reset_index(drop=True)
    elif strategy in (None, "all"):
        df = df.reset_index(drop=True)
    else:
        raise ValueError(f"Unknown sample strategy for {dataset['name']}: {strategy}")

    assert_target_is_last(df, config, f"{dataset['name']} loaded dataset")
    return df


def assert_target_is_last(df, config, context):
    target = config["dataset"]["target"]
    if df.columns[-1] != target:
        raise AssertionError(
            f"{context}: Clusters expects the target/label column last, "
            f"but got last column {df.columns[-1]!r}; expected {target!r}."
        )


def split_dataset(df, config):
    split = config.get("split", {})
    if not split.get("enabled", False):
        return {"all": df.reset_index(drop=True)}

    target = config["dataset"]["target"]
    stratify = df[target] if split.get("stratify", False) else None
    train_df, test_df = train_test_split(
        df,
        test_size=split.get("test_size", 0.25),
        random_state=split.get("random_state", SEED),
        stratify=stratify,
    )
    return {
        "train": train_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }


preview_rows = []
for config in configs:
    df_preview = load_dataset(config)
    preview_rows.append({
        "dataset": config["dataset"]["name"],
        "problem": config["dataset"]["problem"],
        "rows": len(df_preview),
        "columns": len(df_preview.columns),
        "target": config["dataset"]["target"],
    })

pd.DataFrame(preview_rows)

## Run Clusters cases

In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for path in folder.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)


def safe_case_root(config, case_name):
    dataset = config["dataset"]
    output_root = (REPO_ROOT / config["output"]["root"]).resolve()
    case_root = (output_root / dataset["problem"] / dataset["name"] / case_name).resolve()

    if output_root != case_root and output_root not in case_root.parents:
        raise ValueError(f"Refusing to write outside the configured output root: {case_root}")

    return case_root


def run_config(config):
    dataset = config["dataset"]
    if RUN_DATASETS is not None and dataset["name"] not in RUN_DATASETS:
        return []

    df = load_dataset(config)
    splits = split_dataset(df, config)
    records = []

    for case in config["cases"]:
        if RUN_CASES is not None and case["name"] not in RUN_CASES:
            continue

        params = merged_params(config, case)
        model = Clusters(**params)
        case_root = safe_case_root(config, case["name"])

        if config.get("output", {}).get("overwrite", False) and case_root.exists() and not DRY_RUN:
            shutil.rmtree(case_root)

        if DRY_RUN:
            records.append({
                "dataset": dataset["name"],
                "case": case["name"],
                "split": "dry_run",
                "rows": len(df),
                "images": 0,
                "csv_exists": False,
                "output": str(case_root.relative_to(REPO_ROOT)),
            })
            continue

        fit_df = splits["train"] if "train" in splits else next(iter(splits.values()))
        assert_target_is_last(fit_df, config, f"{dataset['name']}/{case['name']} fit data")
        model.fit(fit_df)

        for split_name, split_df in splits.items():
            assert_target_is_last(split_df, config, f"{dataset['name']}/{case['name']}/{split_name} transform data")
            output_dir = case_root / split_name
            model.transform(split_df, str(output_dir))

            generated = count_images(output_dir)
            csv_path = output_dir / f"{params['problem']}.csv"
            if generated != len(split_df):
                raise AssertionError(
                    f"{dataset['name']}/{case['name']}/{split_name}: "
                    f"expected {len(split_df)} images, generated {generated}"
                )
            if not csv_path.exists():
                raise AssertionError(f"Missing route CSV: {csv_path}")

            records.append({
                "dataset": dataset["name"],
                "case": case["name"],
                "algorithm": params["algorithm"],
                "split": split_name,
                "rows": len(split_df),
                "images": generated,
                "csv_exists": csv_path.exists(),
                "output": str(output_dir.relative_to(REPO_ROOT)),
            })

    return records


results = []
for config in configs:
    print(f"Running {config['dataset']['name']} ({config['dataset']['problem']})")
    results.extend(run_config(config))

results_df = pd.DataFrame(results)
results_df

## Summary

In [ ]:
if DRY_RUN:
    summary_df = results_df
else:
    summary_df = (
        results_df
        .groupby(["dataset", "case", "algorithm"], as_index=False)
        .agg(rows=("rows", "sum"), images=("images", "sum"), csv_files=("csv_exists", "sum"))
    )
    if not (summary_df["rows"] == summary_df["images"]).all():
        raise AssertionError("At least one case generated a different number of images than input rows.")

summary_df